# Novel Improvement

## Class Imbalance

### 1. Import Packages and load preprocessed Dataset

In [1]:
#Import an Load the preprocessed DataSet
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, confusion_matrix
)

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

data_dir = PROJECT_ROOT / "data"

X_train = pd.read_csv(data_dir / "X_train_preprocessed.csv")
X_test = pd.read_csv(data_dir / "X_test_preprocessed.csv")
y_train = pd.read_csv(data_dir / "y_train_preprocessed.csv").squeeze()
y_test = pd.read_csv(data_dir / "y_test_preprocessed.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:\n", y_train.value_counts())

X_train shape: (222960, 35)
X_test shape: (55741, 35)
y_train distribution:
 heart_disease_target
False    218247
True       4713
Name: count, dtype: int64


### 2. Importing the eLCS implementation

In [2]:
#Import eLCS

ELCS_ROOT = PROJECT_ROOT / "external" / "scikit-eLCS-master"

if not ELCS_ROOT.exists():
    raise FileNotFoundError(f"Expected the supplied eLCS package at: {ELCS_ROOT}")

if str(ELCS_ROOT) not in sys.path:
    sys.path.insert(0, str(ELCS_ROOT))

from skeLCS import eLCS

### 3. Train the model with & without class imbalance

Train the model on the 10 seeds use for the baseline and produce the measurements (accurancy, precision, recall, F1, balance accuracy, ROC-AUC, PR-AUC, traning seconds, and confusion matrix)

In [3]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_auc_score, average_precision_score

X_train_array = X_train.to_numpy()
X_test_array = X_test.to_numpy()
y_train_array = y_train.to_numpy()
y_test_array = y_test.to_numpy()

print("Arrays ready. X_train_array shape:", X_train_array.shape)

#The seed that are the same as the baseline prediction
SEEDS = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]

#Runs without the class imbalance(SMOTE)
def run_no_smote(seed):
    model = eLCS(random_state=seed)
    start = time.perf_counter()
    model.fit(X_train_array, y_train_array)
    elapsed = time.perf_counter() - start
    preds = model.predict(X_test_array)
    probs = model.predict_proba(X_test_array)[:, 1]
    cm = confusion_matrix(y_test_array, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Seed": seed,
        "Accuracy": accuracy_score(y_test_array, preds),
        "Precision": precision_score(y_test_array, preds, zero_division=0),
        "Recall": recall_score(y_test_array, preds, zero_division=0),
        "F1": f1_score(y_test_array, preds, zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_test_array, preds),
        "ROC-AUC": roc_auc_score(y_test_array, probs),
        "PR-AUC": average_precision_score(y_test_array, probs),
        "Training seconds": elapsed,
        "True negatives": tn, "False positives": fp,
        "False negatives": fn, "True positives": tp,
    }
#Runs with the class imbalance implemented(SMOTE)
def run_smote(seed):
    smote = SMOTE(random_state=seed)
    X_res, y_res = smote.fit_resample(X_train_array, y_train_array)
    model = eLCS(random_state=seed)
    start = time.perf_counter()
    model.fit(X_res, y_res)
    elapsed = time.perf_counter() - start
    preds = model.predict(X_test_array)
    probs = model.predict_proba(X_test_array)[:, 1]
    cm = confusion_matrix(y_test_array, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Seed": seed,
        "Accuracy": accuracy_score(y_test_array, preds),
        "Precision": precision_score(y_test_array, preds, zero_division=0),
        "Recall": recall_score(y_test_array, preds, zero_division=0),
        "F1": f1_score(y_test_array, preds, zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_test_array, preds),
        "ROC-AUC": roc_auc_score(y_test_array, probs),
        "PR-AUC": average_precision_score(y_test_array, probs),
        "Training seconds": elapsed,
        "True negatives": tn, "False positives": fp,
        "False negatives": fn, "True positives": tp,
    }

no_smote_runs = [run_no_smote(s) for s in SEEDS]
no_smote_df = pd.DataFrame(no_smote_runs)
print("No SMOTE - 10 runs:")
display(no_smote_df)

smote_runs = [run_smote(s) for s in SEEDS]
smote_df = pd.DataFrame(smote_runs)
print("With SMOTE - 10 runs:")
display(smote_df)

Arrays ready. X_train_array shape: (222960, 35)
No SMOTE - 10 runs:


,Seed,Accuracy,Precision,Recall,F1,Balanced accuracy,ROC-AUC,PR-AUC,Training seconds,True negatives,False positives,False negatives,True positives
0,3,0.978759,0.350000,0.005942,0.011686,0.502852,0.645900,0.050381,50.210463,54550,13,1171,7
1,6,0.978759,0.000000,0.000000,0.000000,0.499945,0.596128,0.050032,49.680881,54557,6,1178,0
2,9,0.978867,0.000000,0.000000,0.000000,0.500000,0.585835,0.039458,50.278658,54563,0,1178,0
3,12,0.978867,0.000000,0.000000,0.000000,0.500000,0.616573,0.061171,45.238219,54563,0,1178,0
4,15,0.978849,0.000000,0.000000,0.000000,0.499991,0.584002,0.042294,45.519583,54562,1,1178,0
5,18,0.978867,0.000000,0.000000,0.000000,0.500000,0.598950,0.037798,45.142029,54563,0,1178,0
6,21,0.978867,0.000000,0.000000,0.000000,0.500000,0.606452,0.040172,47.328809,54563,0,1178,0
7,24,0.978867,0.000000,0.000000,0.000000,0.500000,0.535399,0.035395,43.577357,54563,0,1178,0
8,27,0.978795,0.166667,0.000849,0.001689,0.500379,0.663726,0.060730,44.049544,54558,5,1177,1
9,30,0.978867,0.000000,0.000000,0.000000,0.500000,0.602241,0.052421,45.612974,54563,0,1178,0


With SMOTE - 10 runs:


,Seed,Accuracy,Precision,Recall,F1,Balanced accuracy,ROC-AUC,PR-AUC,Training seconds,True negatives,False positives,False negatives,True positives
0,3,0.783750,0.087718,0.982173,0.161052,0.880820,0.930587,0.152827,99.707827,42530,12033,21,1157
1,6,0.813817,0.098323,0.955857,0.178306,0.883304,0.937030,0.173939,95.544504,44237,10326,52,1126
2,9,0.826968,0.106369,0.971138,0.191737,0.897496,0.936690,0.164146,95.153062,44952,9611,34,1144
3,12,0.777848,0.085215,0.977080,0.156759,0.875313,0.920132,0.123575,96.218161,42207,12356,27,1151
4,15,0.731347,0.072398,0.991511,0.134943,0.858620,0.920831,0.137576,94.926698,39598,14965,10,1168
5,18,0.850756,0.118740,0.943973,0.210946,0.896358,0.933123,0.165406,96.982984,46310,8253,66,1112
6,21,0.739061,0.074381,0.991511,0.138380,0.862561,0.914619,0.123221,88.944881,40028,14535,10,1168
7,24,0.797061,0.092685,0.978778,0.169335,0.885958,0.927736,0.153238,85.285549,43276,11287,25,1153
8,27,0.817083,0.101185,0.971138,0.183275,0.892447,0.934364,0.161620,90.553980,44401,10162,34,1144
9,30,0.800901,0.094837,0.985569,0.173025,0.891241,0.935786,0.160190,96.509897,43482,11081,17,1161


### 4. Saved the output of the trainning

In [4]:
output_dir = PROJECT_ROOT / "outputs" / "improved_elcs_smote"
output_dir.mkdir(parents=True, exist_ok=True)

no_smote_df.to_csv(output_dir / "no_smote_runs.csv", index=False)
smote_df.to_csv(output_dir / "smote_runs.csv", index=False)

### 5. Summary of the output

In [5]:
summary_cols = ["Accuracy", "Precision", "Recall", "F1", "Balanced accuracy",
                "ROC-AUC", "PR-AUC", "Training seconds"]

no_smote_summary = (
    no_smote_df[summary_cols]
    .agg(["mean", "std", "min", "max"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

smote_summary = (
    smote_df[summary_cols]
    .agg(["mean", "std", "min", "max"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

print("No SMOTE summary (10 runs):")
display(no_smote_summary)

print("\nWith SMOTE summary (10 runs):")
display(smote_summary)

No SMOTE summary (10 runs):


,Metric,mean,std,min,max
0,Accuracy,0.978836,0.000046,0.978759,0.978867
1,Precision,0.051667,0.117181,0.000000,0.350000
2,Recall,0.000679,0.001868,0.000000,0.005942
3,F1,0.001338,0.003675,0.000000,0.011686
4,Balanced accuracy,0.500317,0.000899,0.499945,0.502852
5,ROC-AUC,0.603521,0.034978,0.535399,0.663726
6,PR-AUC,0.046985,0.009335,0.035395,0.061171
7,Training seconds,46.663852,2.545433,43.577357,50.278658



With SMOTE summary (10 runs):


,Metric,mean,std,min,max
0,Accuracy,0.793859,0.037417,0.731347,0.850756
1,Precision,0.093185,0.014140,0.072398,0.118740
2,Recall,0.974873,0.015223,0.943973,0.991511
3,F1,0.169776,0.023304,0.134943,0.210946
4,Balanced accuracy,0.882412,0.013446,0.858620,0.897496
5,ROC-AUC,0.929090,0.007970,0.914619,0.937030
6,PR-AUC,0.151574,0.017679,0.123221,0.173939
7,Training seconds,93.982754,4.354297,85.285549,99.707827
